[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blhuillier/2026B_Galactic_Dynamics/blob/main/notebooks/chapter_04/nb_04_spherical_orbits.ipynb)

# Chapter 4 - Orbits in Spherical Potentials

**Galactic Dynamics · Sejong University · 2026B**

This notebook accompanies the numerical exercises of Chapter 4:

| Part | Exercise | Topic |
|------|----------|-------|
| A | 4.1 | Symplectic versus non-symplectic integration |
| B | 4.2 | Integrating an $N$-body system |

The kick-drift-kick leapfrog is derived in Section 4.8.2.1 of the notes, as
the composition of two exactly solvable Hamiltonian flows. Here it is tested
on the scaled pendulum

$$ H(\theta, p) = \tfrac12 p^2 - \cos\theta , $$

the system of Figure 3.2, with the separatrix at $H = 1$. Part B applies the same
comparison to a self-gravitating system of $N$ stars (Figure 4.6).

Cells marked **TODO** are yours to complete. Everything else is given.


> **Running on Google Colab?** This notebook uses the `galdyn` toolkit from the
> course repository. Run the cell below first - it clones the repo and adds it
> to your Python path. On a local install (via `uv`), this cell does nothing.

In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("2026B_Galactic_Dynamics"):
        !git clone -q https://github.com/blhuillier/2026B_Galactic_Dynamics.git
    sys.path.insert(0, "2026B_Galactic_Dynamics/src")
    %pip install -q astropy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from galdyn.plotting import set_latex_style

set_latex_style(use_tex=False)

rng = np.random.default_rng(20268)

---
## Part A - Exercise 4.1: symplectic versus non-symplectic integration

The pendulum in first-order form is

$$ \dot\theta = p, \qquad \dot p = -\sin\theta . $$

Both integrators below take the same right-hand side and the same step, so any
difference between them is structural, not a matter of accuracy.

In [ ]:
def pendulum_rhs(state):
    """Return d(theta, p)/dt for H = p^2/2 - cos(theta)."""
    theta, p = state
    return np.array([p, -np.sin(theta)])


def pendulum_energy(theta, p):
    """Return H = p^2/2 - cos(theta)."""
    return 0.5 * p**2 - np.cos(theta)


def rk4(state0, dt, n_steps):
    """Classical fourth-order Runge-Kutta. Not symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    for n in range(n_steps):
        y = out[n]
        k1 = pendulum_rhs(y)
        k2 = pendulum_rhs(y + 0.5 * dt * k1)
        k3 = pendulum_rhs(y + 0.5 * dt * k2)
        k4 = pendulum_rhs(y + dt * k3)
        out[n + 1] = y + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
    return out


def leapfrog(state0, dt, n_steps):
    """Second-order kick-drift-kick leapfrog (Section 4.8.2.1). Symplectic."""
    out = np.empty((n_steps + 1, 2))
    out[0] = state0
    theta, p = state0
    for n in range(n_steps):
        p_half = p - 0.5 * dt * np.sin(theta)
        theta = theta + dt * p_half
        p = p_half - 0.5 * dt * np.sin(theta)
        out[n + 1] = (theta, p)
    return out

In [ ]:
state0 = np.array([1.0, 0.0])
dt = 0.05
n_periods = 10_000
n_steps = int(n_periods * 2 * np.pi / dt)

traj_rk4 = rk4(state0, dt, n_steps)
traj_lf = leapfrog(state0, dt, n_steps)
t = dt * np.arange(n_steps + 1)

print(f"{n_steps:,} steps, t_max = {t[-1]:.3g}")

**TODO A.1** - Plot $|H(t) - H(0)|$ against $t$ on logarithmic axes for both
integrators. Which is more accurate over one period? Over $10^4$? Describe the
*shape* of each curve, not just its size.

In [ ]:
# TODO: compute the energy error for each trajectory and plot it.
#
#   err_rk4 = np.abs(pendulum_energy(*traj_rk4.T) - pendulum_energy(*state0))
#   err_lf  = ...
#
# Use ax.loglog, and subsample (e.g. [::200]) so the figure stays light.

**TODO A.2** - Overlay the two trajectories in the $(\theta, p)$ plane. One
stays on a closed curve, the other does not. Connect what you see to
Exercise 3.7 on phase-space volume.

In [ ]:
# TODO: phase-plane overlay. Plot the first ~20 periods and the last ~20
# periods of each trajectory in different colours to make the drift visible.

**TODO A.3** - Halve `dt` and confirm the order of each method: the leapfrog
energy *envelope* should shrink by 4, the Runge-Kutta per-step error by 16.
Does the long-term ranking of the two methods change?

In [ ]:
# TODO: loop over dt in [0.1, 0.05, 0.025, 0.0125], measure
# max|H - H0| over a fixed physical time, and fit the slope on log-log axes.

**TODO A.4** - Repeat with an initial condition just inside the separatrix,
$H(0) = 0.999$. Something goes wrong. Explain why it is not the integrator's
fault. *Hint: Exercise 3.8, part 2.*

In [ ]:
# TODO: choose theta0 = 0 and solve p0 from H0 = 0.999, then integrate.
# Watch whether the orbit stays a libration or escapes into circulation.

**TODO A.5** - The leapfrog does not conserve $H$, but it conserves almost
exactly the shadow Hamiltonian of Section 4.8.2.1. For the pendulum,
$\Phi = -\cos\theta$, it reads

$$ \tilde H = H + \Delta t^2 \left( \tfrac{1}{12} p^2 \cos\theta
- \tfrac{1}{24} \sin^2\theta \right) + \mathcal{O}(\Delta t^4) . $$

Evaluate $\tilde H$ along the leapfrog trajectory and compare its variation
with that of $H$. How does the ratio change when you halve `dt`? Why does the
same test fail for the Runge-Kutta trajectory?


In [ ]:
def shadow_energy(theta, p, dt):
    """Shadow Hamiltonian of the kick-drift-kick leapfrog, to O(dt^2)."""
    raise NotImplementedError("TODO A.5")


# TODO: compare np.ptp(pendulum_energy(*traj_lf.T)) with
# np.ptp(shadow_energy(*traj_lf.T, dt)), and repeat for dt / 2.


---
## Part B - Exercise 4.2: integrating an $N$-body system

The same comparison for a self-gravitating system. $N$ equal-mass stars are drawn from a Plummer sphere in units
$G = M = 1$, $b = 3\pi/16$, in which the crossing time is $2\sqrt2 \simeq 2.8$. Forces are softened,
$\epsilon = 0.1$, and computed by direct summation (Appendix C).

The defaults below ($N = 64$, $T = 200$) run in a few seconds per integrator. Figure 4.6 of the notes uses
$N = 128$ and $T = 500$; try those once your code works.

The sampler, the forces and the diagnostics are given.

In [ ]:
from scipy.integrate import solve_ivp

G = 1.0
N_STARS = 64
EPS = 0.1
B_PLUMMER = 3 * np.pi / 16


def isotropic(n, rng):
    u = rng.normal(size=(n, 3))
    return u / np.linalg.norm(u, axis=1)[:, None]


def plummer_ic(n, rng, b=B_PLUMMER):
    # Radii from M(r); speeds q = v / v_esc from q^2 (1 - q^2)^{7/2}, the isotropic Plummer DF (Chapter 6).
    m = np.full(n, 1.0 / n)
    r = b / np.sqrt(rng.uniform(0, 1, n) ** (-2 / 3) - 1)
    q = np.empty(n)
    k = 0
    while k < n:
        a, y = rng.uniform(0, 1), rng.uniform(0, 0.1)
        if y < a * a * (1 - a * a) ** 3.5:
            q[k] = a
            k += 1
    x = r[:, None] * isotropic(n, rng)
    v = (q * np.sqrt(2 * G / np.sqrt(r**2 + b**2)))[:, None] * isotropic(n, rng)
    x -= (m[:, None] * x).sum(0)
    v -= (m[:, None] * v).sum(0)
    return m, x, v


def accel(x, m, eps=EPS):
    # Softened accelerations by direct summation: a_i = G sum_j m_j (x_j - x_i) / (|x_j - x_i|^2 + eps^2)^{3/2}.
    dx = x[:, 0] - x[:, 0, None]
    dy = x[:, 1] - x[:, 1, None]
    dz = x[:, 2] - x[:, 2, None]
    w = (dx * dx + dy * dy + dz * dz + eps * eps) ** -1.5 * m
    np.fill_diagonal(w, 0.0)
    return G * np.stack([(w * dx).sum(1), (w * dy).sum(1), (w * dz).sum(1)], axis=1)


def total_energy(x, v, m, eps=EPS):
    K = 0.5 * np.sum(m * np.sum(v**2, axis=1))
    d2 = ((x[None] - x[:, None]) ** 2).sum(-1) + eps**2
    U = -G * np.sum(np.triu(m[:, None] * m[None, :] / np.sqrt(d2), 1))
    return K + U


def angular_momentum(x, v, m):
    return (m[:, None] * np.cross(x, v)).sum(0)


def diagnostics(out, m):
    # out: list of (t, x, v). Returns t, |dE/E0| and |dL| / sum_i m_i |x_i x v_i|.
    t = np.array([o[0] for o in out])
    E = np.array([total_energy(xx, vv, m) for _, xx, vv in out])
    L = np.array([angular_momentum(xx, vv, m) for _, xx, vv in out])
    Ls = np.sum(m * np.linalg.norm(np.cross(out[0][1], out[0][2]), axis=1))
    return t, np.abs(E / E[0] - 1), np.linalg.norm(L - L[0], axis=1) / Ls


m_nb, x_nb, v_nb = plummer_ic(N_STARS, np.random.default_rng(2026))
print("E0 =", total_energy(x_nb, v_nb, m_nb))

**TODO B.1** - Implement the two integrators for $N$ bodies. Both return a list of snapshots `(t, x, v)` and
the number of force evaluations. `run_symplectic` applies, at each step, one kick-drift-kick sub-step of length
$c\,\Delta t$ for each coefficient $c$ in `coeffs`: `LEAPFROG` gives the leapfrog, `YOSHIDA` the fourth-order
composition. Reuse the last acceleration of one sub-step for the first half kick of the next.

In [ ]:
W1 = 1 / (2 - 2 ** (1 / 3))
W0 = 1 - 2 * W1
LEAPFROG = (1.0,)
YOSHIDA = (W1, W0, W1)


def run_symplectic(x, v, m, dt, T, coeffs=LEAPFROG, nout=200):
    # TODO B.1
    raise NotImplementedError("TODO B.1")


def run_rk4_nbody(x, v, m, dt, T, nout=200):
    # TODO B.1: pack (x, v) into one vector y and reuse the RK4 step of Part A.
    raise NotImplementedError("TODO B.1")

**TODO B.2** - Run the leapfrog ($\Delta t = 0.01$), Yoshida ($\Delta t = 0.03$) and RK4 ($\Delta t = 0.04$),
which all cost about 100 force evaluations per unit time. Plot $|\Delta E/E_0|$ and the angular-momentum error
against $t$ on logarithmic axes. Which scheme is best for the energy? For the angular momentum? Is the
fourth-order symplectic scheme better than the leapfrog at equal cost?

In [ ]:
T_NB = 200.0
# TODO B.2: out, nf = run_symplectic(x_nb, v_nb, m_nb, 0.01, T_NB); t, dE, dL = diagnostics(out, m_nb); ...

**TODO B.3** - Add `scipy`'s adaptive eighth-order Runge-Kutta, `solve_ivp(..., method="DOP853")`, at
`rtol = 1e-5` and `1e-7` (with `atol = 1e-2 * rtol`). Use `t_eval` to get snapshots and `sol.nfev` for the
number of force evaluations. How do its cost and its errors compare?

In [ ]:
# TODO B.3

**TODO B.4** - Show on paper that each kick and each drift conserves the total linear and angular momenta
exactly for pairwise central forces. Which of the two does RK4 conserve exactly? Check numerically.

*Your answer here.*

**TODO B.5** - Perturb one coordinate of one star by $10^{-10}$, rerun the leapfrog to $t = 60$, and plot the
rms distance between the two runs on a logarithmic axis. Fit the e-folding time and compare it with the
crossing time. Do the same for the leapfrog and RK4 runs started from identical initial conditions with the
same step. What does this imply about comparing trajectories?

In [ ]:
# TODO B.5